# Change Data Capture (CDC) Lab

This notebook demonstrates Delta Lake Change Data Feed (CDF) — enabling CDF on a source table, simulating business changes (INSERT, UPDATE, DELETE), auditing the change log with `table_changes()`, and incrementally syncing a target table via `MERGE INTO` (SCD Type 1).

## 1. Create Source Table with Change Data Feed

Create a source table with `delta.enableChangeDataFeed = true` and insert initial movie records. This flag tells Delta Lake to capture every row-level change (insert, update, delete) in a change log.

In [0]:
%sql
CREATE OR REPLACE TABLE main.lab_data.cdc_source_movies (
  id INT,
  title STRING,
  budget_millions DECIMAL(10,2),
  revenue_millions DECIMAL(10,2),
  rating DECIMAL(3,1)
) TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

INSERT INTO main.lab_data.cdc_source_movies VALUES
  (1, 'Inception', 160.00, 836.80, 8.8),
  (2, 'The Dark Knight', 185.00, 1005.00, 9.0),
  (3, 'Interstellar', 165.00, 701.70, 8.6);

SELECT * FROM main.lab_data.cdc_source_movies ORDER BY id;

## 2. Create Target Table

Create the target table that will receive replicated data, then perform an initial full load from the source.

In [0]:
%sql
CREATE OR REPLACE TABLE main.lab_data.cdc_target_movies (
  id INT,
  title STRING,
  budget_millions DECIMAL(10,2),
  revenue_millions DECIMAL(10,2),
  rating DECIMAL(3,1)
);

-- Initial full load
INSERT INTO main.lab_data.cdc_target_movies
SELECT * FROM main.lab_data.cdc_source_movies;

SELECT * FROM main.lab_data.cdc_target_movies ORDER BY id;

## 3. Simulate Business Changes

Apply three types of changes to the source table to simulate real-world data operations:

* **INSERT** — a new movie is added
* **UPDATE** — revenue is corrected for an existing movie
* **DELETE** — a movie is removed

In [0]:
%sql
-- INSERT: new movie added to the catalog
INSERT INTO main.lab_data.cdc_source_movies VALUES
  (4, 'Avatar', 237.00, 2923.70, 7.9);

-- UPDATE: corrected revenue for Inception
UPDATE main.lab_data.cdc_source_movies
SET revenue_millions = 900.00
WHERE id = 1;

-- DELETE: remove Interstellar from the catalog
DELETE FROM main.lab_data.cdc_source_movies
WHERE id = 3;

SELECT * FROM main.lab_data.cdc_source_movies ORDER BY id;

## 4. Audit the Change Log with table_changes()

Query `table_changes()` to inspect the full transaction log. Databricks automatically exposes technical columns:

* `_change_type` — `insert`, `update_preimage` (before), `update_postimage` (after), `delete`
* `_commit_version` — Delta Lake commit version of the change
* `_commit_timestamp` — when the change was committed

In [0]:
%sql
SELECT 
  _change_type,
  _commit_version,
  _commit_timestamp,
  id,
  title,
  budget_millions,
  revenue_millions,
  rating
FROM table_changes('main.lab_data.cdc_source_movies', 0)
ORDER BY _commit_version, id;

## 5. Incremental MERGE INTO (SCD Type 1)

Sync the target table by processing only the changes since the initial load (version 1). The `MERGE INTO` handles:

* **New rows** (`insert`) → insert into target
* **Updated rows** (`update_postimage`) → overwrite with latest values
* **Deleted rows** (`delete`) → remove from target

This is SCD Type 1: only the current state is kept, no history is retained.

In [0]:
%sql
MERGE INTO main.lab_data.cdc_target_movies AS t
USING (
  SELECT 
    id, title, budget_millions, revenue_millions, rating, _change_type
  FROM table_changes('main.lab_data.cdc_source_movies', 1)
  WHERE _change_type IN ('insert', 'update_postimage', 'delete')
) AS s
ON t.id = s.id
WHEN MATCHED AND s._change_type = 'delete' THEN DELETE
WHEN MATCHED AND s._change_type = 'update_postimage' THEN
  UPDATE SET 
    title = s.title,
    budget_millions = s.budget_millions,
    revenue_millions = s.revenue_millions,
    rating = s.rating
WHEN NOT MATCHED AND s._change_type = 'insert' THEN
  INSERT (id, title, budget_millions, revenue_millions, rating)
  VALUES (s.id, s.title, s.budget_millions, s.revenue_millions, s.rating);

-- Verify the synced target table
SELECT * FROM main.lab_data.cdc_target_movies ORDER BY id;

## Key Takeaways

* **Change Data Feed (CDF):** Enabling `delta.enableChangeDataFeed = true` on the source table captures every row-level change with metadata columns `_change_type`, `_commit_version`, and `_commit_timestamp`.
* **table_changes() function:** Provides an audit trail of all modifications — `insert`, `update_preimage` (before), `update_postimage` (after), and `delete` — enabling point-in-time reconstruction of the table state.
* **MERGE INTO for SCD Type 1:** The incremental sync uses `table_changes()` filtered to `insert`, `update_postimage`, and `delete` events. New rows are inserted, existing rows are overwritten with latest values, and deleted rows are removed from the target — maintaining only the current state.
* **Production pattern:** In a real pipeline, the `since_version` parameter would be tracked between runs (e.g., stored in a metadata table) so each MERGE only processes new changes since the last sync.